# MANATUABON PHASE 9: KAGGLE LORA FINE-TUNING

## NVIDIA Nemotron-3-Nano-30B Reasoning Optimization
Run this notebook on **Google Colab Pro** with an **A100 or L4 GPU**.
Make sure to upload `synthetic_cot.jsonl` (generated by `kaggle_pipeline.py`) to the Colab workspace before running.

In [1]:
# 1. Install Required Dependencies (Including Mamba-SSM for Hybrid Architecture)
!pip install -U transformers peft accelerate bitsandbytes datasets trl mamba-ssm causal-conv1d

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 7.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 168.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 kB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.2 MB/s eta 0:00:00
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-

In [2]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

def format_prompt(sample):
    """
    Format the ShareGPT conversational format generated by kaggle_pipeline.py
    into a continuous prompt string for Next-Token Prediction.
    """
    convo = sample["conversations"]
    prompt = convo[0]["value"]
    response = convo[1]["value"]

    # Simple template: User prompt + Assistant reasoning and \boxed{} answer
    text = f"User: {prompt}\n\nAssistant: {response}"
    return {"text": text}

# 2. Load the generated dataset (Upload synthetic_cot.jsonl to your Colab workspace)
dataset_path = "synthetic_cot.jsonl"
if not os.path.exists(dataset_path):
    print(f"\n❌ ERROR: Please upload {dataset_path} to your Colab workspace!")
else:
    dataset = load_dataset("json", data_files=dataset_path, split="train")
    dataset = dataset.map(format_prompt)

    # Create a 90/10 Train/Eval split for Early Stopping
    dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
    train_data = dataset_split["train"]
    eval_data = dataset_split["test"]
    print(f"\n✅ Loaded {len(train_data)} training and {len(eval_data)} validation traces.")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/95 [00:00<?, ? examples/s]


✅ Loaded 85 training and 10 validation traces.


In [3]:
# 3. Configure 4-bit Quantization to fit 30B in VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 4. Load the Nemotron 30B Nano Model & Tokenizer
model_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16" # Corrected public repo ID
print(f"Loading base model: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

Loading base model: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16...


config.json: 0.00B [00:00, ?B/s]

configuration_nemotron_h.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16:
- configuration_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16:
- modeling_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/197 [00:00<?, ?B/s]

In [4]:
# 5. Configure the LoRA Adapter (Rank 32 maximum per Kaggle rules)
print("Configuring Rank-32 LoRA adapter...")
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Configuring Rank-32 LoRA adapter...
trainable params: 869,318,656 || all params: 32,447,256,000 || trainable%: 2.6792


In [5]:
# 6. Set up the SFTTrainer with Early Stopping
output_dir = "./nemotron_reasoning_lora"
sft_config = SFTConfig(
    output_dir=output_dir,
    dataset_text_field="text", # Explicit text map for dataset
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=150, # Set a high limit, early stopping will halt it sooner if needed
    learning_rate=2e-4,
    fp16=True, # A100 supports bf16, T4 uses fp16
    logging_steps=5,
    eval_strategy="steps", # Required for early stopping
    eval_steps=10,         # Check validation loss every 10 steps
    save_strategy="steps",
    save_steps=10,
    load_best_model_at_end=True, # Loads the model from the step with highest eval score
    metric_for_best_model="eval_loss",
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    max_seq_length=4096, # High sequence length required for <think> CoT traces
    tokenizer=tokenizer,
    args=sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Halts if eval loss goes up 3 times in a row
)

# 7. Train the model
print("\n🔥 Starting LoRA Fine-tuning with Early Stopping...")
trainer.train()

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'max_seq_length'

In [ ]:
# 8. Save the final adapter for Kaggle submission
print(f"\n✅ Training complete. Saving LoRA adapter to {output_dir}")
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Adapter is ready to be zipped and submitted to Kaggle!")